# T326 — Ciência de Dados | Trabalho 1
## 01 — preparação dos dados: preparação e integração dos dados

**Universidade de Fortaleza — Equipe: Luma, Amanda, Peter e Luís.**

Pergunta-guia do trabalho: **“O Brasil é um país desigual?”**

Esta etapa prepara uma base municipal reproduzível do **Sudeste (ES, MG, RJ e SP)**. Seu objetivo é verificar fontes, cobertura, unidades e qualidade e calcular PIB per capita. As análises econômicas, rankings e gráficos dos colegas não são implementados aqui. A preparação sozinha não responde à pergunta-guia.

## 2. Fontes e referência temporal

Arquivos fornecidos pelo professor, obtidos pelos IDs de Google Drive:

| Fonte | ID | Nome original verificado no Drive |
|---|---|---|
| PIB municipal — SIDRA | `1GQVxYHY9ouZvh_Jkbzt3sjnWHslxGqnX` | `tabela5938_2020.csv.xz` |
| Estimativa populacional — SIDRA | `1OLg85S7vAr4MQomc_wcaQhRo6SLciSu0` | `tabela6579_2020.csv.xz` |
| DTB | `1G3Ll5LsIhsvpodjnKg6JE0KXbc_D-BSP` | `RELATORIO_DTB_BRASIL_MUNICIPIO.csv` |

PIB e população não contêm coluna de ano: **2020 vem dos nomes originais**, confirmados pelo cabeçalho HTTP `Content-Disposition` e pelos títulos do Drive. A evidência e os hashes estão em `data/raw/downloads.jsonl` e `config.json`. A DTB não informa sua edição; essa limitação será mantida no relatório.

Metadados de referência: [SIDRA 5938](https://sidra.ibge.gov.br/tabela/5938), [SIDRA 6579](https://sidra.ibge.gov.br/tabela/6579), [DTB](https://www.ibge.gov.br/geociencias/organizacao-do-territorio/divisao-regional/23701-divisao-territorial-brasileira.html), [códigos municipais](https://www.ibge.gov.br/explica/codigos-dos-municipios.php). Essas páginas contextualizam as fontes; não substituem os arquivos fornecidos.

## 3. Bibliotecas e carregamento

Execute com o Python do ambiente `.venv`. A raiz é encontrada por `config.json` e `src/sudeste`, permitindo iniciar o kernel na raiz ou na pasta de notebooks. O notebook não depende de caminhos absolutos nem de acesso à rede após o download.

In [1]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# Localização portátil da raiz do projeto.
RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'config.json').is_file() and (p / 'src/sudeste').is_dir()), None)
if RAIZ is None:
    raise FileNotFoundError('Execute este notebook dentro da pasta do projeto.')
if str(RAIZ / 'src') not in sys.path:
    sys.path.insert(0, str(RAIZ / 'src'))

from sudeste.io import carregar_fontes
from sudeste.pipeline import normalizar, integrar, validar_final, exportar
from sudeste.preparacao import filtrar_sudeste, selecionar_ano

CONFIG = json.loads((RAIZ / 'config.json').read_text(encoding='utf-8'))
# Para escolher outro ano, ele precisa existir em AMBAS as fontes.
# CONFIG['ano'] = 2020
bases, inventario = carregar_fontes(RAIZ, CONFIG)
display(pd.DataFrame({n: {k: i.get(k) for k in
    ['nome_original', 'formato', 'compactacao', 'encoding', 'registros', 'ano_metadados']}
    for n, i in inventario.items()}).T)

,nome_original,formato,compactacao,encoding,registros,ano_metadados
pib,tabela5938_2020.csv.xz,csv,xz,utf-8,5570,2020
populacao,tabela6579_2020.csv.xz,csv,xz,utf-8,5570,2020
dtb,RELATORIO_DTB_BRASIL_MUNICIPIO.csv,csv,None,utf-8,5570,None


## 4. Inspeção inicial

A leitura identifica XZ pela assinatura do arquivo, descompacta em memória e preserva os originais. Os três conteúdos reais são CSV UTF-8, separados por vírgula e com cabeçalho na primeira linha; não há abas. Inicialmente, as colunas são lidas como texto para preservar códigos geográficos e marcadores de ausência. O inventário registra também tipos inferidos, colunas, vazios e duplicatas.

In [2]:
for nome, dados in bases.items():
    display(Markdown(f'### Fonte: {nome} — {len(dados)} registros'))
    display(dados.head(3))
    display(pd.DataFrame({
        'tipo_leitura': dados.dtypes.astype(str),
        'vazios': dados.apply(lambda s: s.str.strip().eq('').sum()),
    }))
    print('Duplicatas exatas:', dados.duplicated().sum())

### Fonte: pib — 5570 registros

,Cód.,Município,PIB (Mil Reais),Impostos (Mil Reais),VA da agropecuária (Mil Reais),VA da indústria (Mil Reais),VA dos serviços (Mil Reais),VA da administração pública (Mil Reais)
0,1100015,Alta Floresta D'Oeste (RO),570272.0,35109.0,203394.0,20716.0,150192.0,160860.0
1,1100023,Ariquemes (RO),2818049.0,295656.0,199723.0,404752.0,1207405.0,710513.0
2,1100031,Cabixi (RO),167190.0,7237.0,81177.0,5438.0,28667.0,44671.0


,tipo_leitura,vazios
Cód.,string,0
Município,string,0
PIB (Mil Reais),string,0
Impostos (Mil Reais),string,0
VA da agropecuária (Mil Reais),string,0
VA da indústria (Mil Reais),string,0
VA dos serviços (Mil Reais),string,0
VA da administração pública (Mil Reais),string,0


Duplicatas exatas: 0


### Fonte: populacao — 5570 registros

...

Duplicatas exatas: 0


### Fonte: dtb — 5570 registros

,UF,Nome_UF,Região Geográfica Intermediária,Nome Região Geográfica Intermediária,Região Geográfica Imediata,Nome Região Geográfica Imediata,Mesorregião Geográfica,Nome_Mesorregião,Microrregião Geográfica,Nome_Microrregião,Município,Código Município Completo,Nome_Município
0,11,Rondônia,1102,Ji-Paraná,110005,Cacoal,02,Leste Rondoniense,006,Cacoal,00015,1100015,Alta Floresta D'Oeste
1,11,Rondônia,1102,Ji-Paraná,110005,Cacoal,02,Leste Rondoniense,006,Cacoal,00379,1100379,Alto Alegre dos Parecis
2,11,Rondônia,1101,Porto Velho,110002,Ariquemes,02,Leste Rondoniense,003,Ariquemes,00403,1100403,Alto Paraíso


,tipo_leitura,vazios
UF,string,0
Nome_UF,string,0
Região Geográfica Intermediária,string,0
Nome Região Geográfica Intermediária,string,0
Região Geográfica Imediata,string,0
Nome Região Geográfica Imediata,string,0
Mesorregião Geográfica,string,0
Nome_Mesorregião,string,0
Microrregião Geográfica,string,0
Nome_Microrregião,string,0


Duplicatas exatas: 0


## 5. Padronização e limpeza

A chave é `Cód.` nos arquivos SIDRA e `Código Município Completo` na DTB. A coluna `Município` da DTB é um código local de cinco dígitos e **não** é usada isoladamente no cruzamento. Códigos completos ficam como texto de sete dígitos; não completamos códigos truncados.

Os nomes das colunas recebem `snake_case` sem acentos. PIB, impostos e Valores Adicionados são multiplicados por **1.000**, conforme os cabeçalhos em mil reais. População é medida em pessoas. Códigos inválidos e chaves duplicadas interrompem a execução com auditoria; não há remoção arbitrária de duplicatas.

In [3]:
tratadas = normalizar(bases, CONFIG, RAIZ)
display(tratadas['pib'][['codigo_municipio', 'ano', 'pib_mil_reais', 'pib_total_reais']].head())
display(tratadas['dtb'][['codigo_municipio_local', 'codigo_municipio', 'municipio', 'codigo_uf', 'estado']].head())

,codigo_municipio,ano,pib_mil_reais,pib_total_reais
0,1100015,2020,570272.0,570272000.0
1,1100023,2020,2818049.0,2818049000.0
2,1100031,2020,167190.0,167190000.0
3,1100049,2020,2519353.0,2519353000.0
4,1100056,2020,600670.0,600670000.0


,codigo_municipio_local,codigo_municipio,municipio,codigo_uf,estado
0,00015,1100015,Alta Floresta D'Oeste,11,Rondônia
1,00379,1100379,Alto Alegre dos Parecis,11,Rondônia
2,00403,1100403,Alto Paraíso,11,Rondônia
3,00346,1100346,Alvorada D'Oeste,11,Rondônia
4,00023,1100023,Ariquemes,11,Rondônia


## 6. DTB e filtro do Sudeste

O código de UF da DTB determina o recorte: `31 → MG`, `32 → ES`, `33 → RJ`, `35 → SP`. A região é adicionada por essa correspondência documentada; regiões geográficas imediatas e intermediárias da DTB não são confundidas com as Grandes Regiões. O total municipal esperado nesta execução é obtido da própria DTB fornecida.

In [4]:
territorio = filtrar_sudeste(tratadas['dtb'])
display(territorio.groupby(['uf', 'estado']).size().rename('municipios_na_dtb').to_frame())
print('Cobertura territorial do recorte:', len(territorio))

,,municipios_na_dtb
uf,estado,
ES,Espírito Santo,78
MG,Minas Gerais,853
RJ,Rio de Janeiro,92
SP,São Paulo,645


Cobertura territorial do recorte: 1668


## 7. Compatibilidade temporal

A seleção utiliza o ano mais recente na interseção entre PIB e população. `config.json` torna essa escolha explícita e configurável. Arquivos sem coluna de ano só podem ser utilizados com ano e evidência documentados; os hashes evitam aplicar silenciosamente os metadados de 2020 a arquivos modificados.

In [5]:
ano, anos_comuns = selecionar_ano(tratadas['pib'], tratadas['populacao'], CONFIG.get('ano'))
print('Anos do PIB:', sorted(tratadas['pib']['ano'].unique()))
print('Anos da população:', sorted(tratadas['populacao']['ano'].unique()))
print('Anos comuns:', anos_comuns)
print('Ano selecionado:', ano)
print('Edição DTB:', CONFIG['fontes']['dtb']['evidencia_ano'])

Anos do PIB: [np.int64(2020)]
Anos da população: [np.int64(2020)]
Anos comuns: [2020]
Ano selecionado: 2020
Edição DTB: Edição não informada no conteúdo nem no nome original; não presumir 2020.


## 8. Cruzamentos e cálculo do PIB per capita

Os cruzamentos são validados como um para um, mantendo inicialmente todos os municípios da DTB do Sudeste. Uma auditoria nacional externa registra chaves sem correspondência em qualquer lado; auditorias regionais registram ausências de PIB e população.

\[
PIB\ per\ capita\;(R\$/pessoa)=\frac{PIB\ municipal\;(R\$)}{população\ municipal\;(pessoas)}
\]

População nula, zero, negativa, fracionária ou não finita e PIB nulo, negativo ou não finito impedem o cálculo: esses registros ficam em `registros_excluidos.csv`, sem imputação. Valores Adicionados e impostos não finitos viram nulos auditados; negativos são preservados e sinalizados, pois exigem interpretação da variável.

In [6]:
base_final, qualidade = integrar(tratadas, CONFIG, RAIZ, inventario)
# Conferência independente da fórmula, sem arredondamento prévio.
np.testing.assert_allclose(
    base_final['pib_per_capita_reais'].to_numpy(dtype=float),
    (base_final['pib_total_reais'] / base_final['populacao']).to_numpy(dtype=float),
    rtol=1e-12, atol=1e-8,
)
display(base_final[['codigo_municipio', 'municipio', 'uf', 'ano',
                    'pib_total_reais', 'populacao', 'pib_per_capita_reais']].head())

,codigo_municipio,municipio,uf,ano,pib_total_reais,populacao,pib_per_capita_reais
0,3200102,Afonso Cláudio,ES,2020,498896000.0,30455,16381.415203
1,3200136,Águia Branca,ES,2020,192510000.0,9631,19988.578548
2,3200169,Água Doce do Norte,ES,2020,165981000.0,10909,15215.051792
3,3200201,Alegre,ES,2020,504636000.0,29975,16835.229358
4,3200300,Alfredo Chaves,ES,2020,370237000.0,14636,25296.324132


## 9. Validações e relatório de qualidade

Validamos chave município/ano, região, ano, códigos, positividade da população, unidades, fórmula e reconciliação das exclusões. As contagens por UF abaixo são de cobertura, não análises econômicas. As categorias de perdas podem se sobrepor; o total de exclusões é contado uma única vez.

In [7]:
validar_final(base_final, qualidade)
display(pd.Series(qualidade['etapas'], name='registros').to_frame())
display(pd.Series(qualidade['perdas_e_recortes'], name='registros').to_frame())
display(pd.DataFrame({
    'dtb': qualidade['registros_por_uf_dtb'],
    'base_final': qualidade['registros_por_uf_final'],
}))
display(pd.Series(qualidade['ausentes_final'], name='ausentes').to_frame())
print('Duplicatas município/ano:', qualidade['duplicatas_chave_final'])
print('Divergências de nomes:', qualidade['divergencias_nomes'])

,registros
pib_bruto,5570
populacao_bruta,5570
dtb_bruta,5570
pib_ano_selecionado,5570
populacao_ano_selecionado,5570
dtb_sudeste,1668
pib_sudeste,1668
populacao_sudeste,1668
apos_cruzar_pib,1668
apos_cruzar_populacao,1668


,registros
pib_outros_anos,0
populacao_outros_anos,0
dtb_fora_sudeste,3902
pib_sem_dtb,0
pib_fora_sudeste_com_dtb,3902
populacao_sem_dtb,0
populacao_fora_sudeste_com_dtb,3902
dtb_sudeste_sem_pib,0
dtb_sudeste_sem_populacao,0
excluidos_calculo_invalido,0


,dtb,base_final
ES,78,78
MG,853,853
RJ,92,92
SP,645,645


,ausentes
codigo_municipio,0
municipio,0
codigo_uf,0
uf,0
estado,0
regiao,0
ano,0
pib_total_reais,0
populacao,0
pib_per_capita_reais,0


Duplicatas município/ano: 0
Divergências de nomes: 0


## 10. Exportação da base tratada

CSV UTF-8 e Parquet contêm o mesmo conjunto de registros; o Parquet preserva tipos. O exportador relê ambos, compara valores e tipos e gera metadados e relatório Markdown/JSON. Na leitura do CSV, informe `dtype` para os códigos.

In [8]:
caminho_csv, caminho_parquet = exportar(base_final, qualidade, RAIZ)
print('CSV:', caminho_csv.relative_to(RAIZ))
print('Parquet:', caminho_parquet.relative_to(RAIZ))
print('Releitura:', qualidade['releitura'])
print('Relatório: outputs/reports/qualidade.md')
# Exemplo de consumo para a equipe.
dados_equipe = pd.read_parquet(caminho_parquet)
assert len(dados_equipe) == len(base_final)

CSV: data/processed/sudeste_municipios.csv
Parquet: data/processed/sudeste_municipios.parquet
Releitura: {'csv': 'aprovada', 'parquet': 'aprovada'}
Relatório: outputs/reports/qualidade.md


## 11. Conclusões parciais e limitações

O texto a seguir é construído a partir dos resultados efetivamente executados. Não há conclusão sobre desigualdade econômica nesta etapa.

In [9]:
cobertura = ', '.join(f'{uf}: {q}' for uf, q in sorted(qualidade['registros_por_uf_final'].items()))
excluidos = qualidade['perdas_e_recortes']['excluidos_calculo_invalido']
ausentes = sum(qualidade['ausentes_final'].values())
display(Markdown(
    f'Foram disponibilizados **{len(base_final)} municípios** no ano **{ano}** ({cobertura}). '
    f'O recorte territorial incluiu {qualidade["etapas"]["dtb_sudeste"]} municípios na DTB; '
    f'{excluidos} registros foram excluídos por inviabilidade do cálculo. '
    f'A base final apresenta {qualidade["duplicatas_chave_final"]} duplicatas de município/ano '
    f'e {ausentes} valores ausentes no conjunto de colunas exportadas. '
    'As exportações CSV e Parquet passaram pela releitura.\n\n'
    'O ano econômico e populacional foi identificado nos nomes originais dos arquivos, '
    'não em uma coluna temporal. A edição da DTB permanece desconhecida, portanto a correspondência '
    'de códigos não comprova que limites territoriais sejam historicamente idênticos. '
    'Os valores monetários são correntes e o PIB per capita não equivale a renda pessoal.\n\n'
    'A base está preparada para as contribuições reais de estatística descritiva, análises econômicas e distribuição e síntese. '
    'A futura consolidação exige seus notebooks, execução conjunta do início ao fim e revisão técnica por preparação dos dados.'
))

Foram disponibilizados **1668 municípios** no ano **2020** (ES: 78, MG: 853, RJ: 92, SP: 645). O recorte territorial incluiu 1668 municípios na DTB; 0 registros foram excluídos por inviabilidade do cálculo. A base final apresenta 0 duplicatas de município/ano e 0 valores ausentes no conjunto de colunas exportadas. As exportações CSV e Parquet passaram pela releitura.

O ano econômico e populacional foi identificado nos nomes originais dos arquivos, não em uma coluna temporal. A edição da DTB permanece desconhecida, portanto a correspondência de códigos não comprova que limites territoriais sejam historicamente idênticos. Os valores monetários são correntes e o PIB per capita não equivale a renda pessoal.

A base está preparada para as contribuições reais de estatística descritiva, análises econômicas e distribuição e síntese. A futura consolidação exige seus notebooks, execução conjunta do início ao fim e revisão técnica por preparação dos dados.